# 🐍 Portuguese Code Model Fine-tuning

This notebook demonstrates how to fine-tune the Qwen2.5-Coder-0.5B model for Portuguese code generation.

## Setup

In [ ]:
# Install dependencies (if needed)
# !pip install unsloth transformers datasets torch

In [ ]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
import json

# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Load the Base Model

In [ ]:
model_name = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    load_in_4bit=True,
)

print(f"✅ Loaded {model_name}")
print(f"   Parameters: ~0.5B")
print(f"   Max sequence length: {max_seq_length}")

## 2. Test Base Model (Before Fine-tuning)

In [ ]:
def generate_response(model, tokenizer, prompt, max_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.2,
        do_sample=True,
        top_p=0.95,
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# Test with a Portuguese prompt
test_prompt = """### Instrução:
Escreva uma função em Python para calcular o fatorial

### Resposta:
"""

print("📝 Testing base model:")
print("-" * 50)
response = generate_response(model, tokenizer, test_prompt)
print(response[len(test_prompt):])

## 3. Prepare Dataset

In [ ]:
# Load sample data
dataset = load_dataset("json", data_files="../data/sample_data.jsonl", split="train")

print(f"📊 Dataset size: {len(dataset)} examples")

# Preview first example
print("\n📝 First example:")
print(json.dumps(dataset[0], indent=2, ensure_ascii=False))

In [ ]:
# Format dataset
def format_prompt(example):
    instruction = example["instruction"]
    input_text = example.get("input", "")
    output = example["output"]
    
    if input_text:
        prompt = f"### Instrução:\n{instruction}\n\n### Entrada:\n{input_text}\n\n### Resposta:\n{output}"
    else:
        prompt = f"### Instrução:\n{instruction}\n\n### Resposta:\n{output}"
    
    return {"text": prompt}

formatted_dataset = dataset.map(format_prompt)

# Show formatted example
print("Formatted example:")
print("-" * 50)
print(formatted_dataset[0]["text"][:500] + "...")

## 4. Add LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

print("✅ LoRA adapters added")
model.print_trainable_parameters()

## 5. Train the Model

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)

In [ ]:
# Start training
print("🚀 Starting training...")
trainer.train()
print("✅ Training complete!")

## 6. Test Fine-tuned Model

In [ ]:
# Test with same prompt
print("📝 Testing fine-tuned model:")
print("-" * 50)
response = generate_response(model, tokenizer, test_prompt)
print(response[len(test_prompt):])

## 7. Save Model

In [ ]:
output_dir = "../model_output"

# Save merged model (LoRA + base)
model.save_pretrained_merged(output_dir, tokenizer, save_method="merged_16bit")

print(f"✅ Model saved to {output_dir}")

## 8. Export to ONNX (for transformers.js)

In [ ]:
# Run the export script
!python ../export_onnx.py --model_path ../model_output --output_dir ../onnx_model

## 🎉 Done!

Your fine-tuned model is ready for use with transformers.js!